In [ ]:
from pathlib import Path
import pandas as pd

filename = "job_postings_train.csv"
candidates = [
    Path.cwd() / filename,
    Path.home() / "Downloads" / filename,
    Path("/kaggle/input/competitions/fake-job-posting") / filename,
]

data_path = next((path for path in candidates if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError(
        f"Could not find {filename}. Checked:\n"
        + "\n".join(str(path) for path in candidates)
    )

df = pd.read_csv(data_path)
print(f"Loaded {data_path} — shape: {df.shape}")

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print(f"Training data shape: {df.shape}")

In [ ]:
df.info()

In [ ]:
print(df.shape)

In [ ]:
print(df.describe().T) 

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
print(df.describe(include='object'))

In [ ]:
# Create the cleaned dataset
df_clean = df.copy()

# Convert object columns to category
df_clean[df_clean.select_dtypes('object').columns] = (
    df_clean.select_dtypes('object').astype('category')
)

# Verify conversion
print(df_clean.dtypes)
print("Success:", not any(df_clean.dtypes == 'object'))

In [ ]:
print(df.describe(include='number').T)  

In [ ]:
print(df.duplicated().sum())

In [ ]:
for i in df.columns:
    col_num = df[i].isnull().sum()
    na_per = (col_num / df.shape[0]) * 100
    print(f"Missing in {i} is: {na_per:.2f}%")

In [ ]:
import pandas as pd
import numpy as np

# Check missing values
print(df.isnull().sum())

# Check percentage of missing values
missing_percent = (df.isnull().sum() / len(df)) * 100
print(missing_percent.sort_values(ascending=False))

In [ ]:
import pandas as pd
import numpy as np

# Make a copy of the original dataset
df_clean = df.copy()

# 1. Fill missing categorical and text columns
text_cols = [
    'title',
    'location',
    'department',
    'salary_range',
    'company_profile',
    'description',
    'requirements',
    'benefits',
    'employment_type',
    'required_experience',
    'required_education',
    'industry',
    'function'
]

for col in text_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna('Unknown')

# 2. Fill missing numerical columns with the median
numeric_cols = df_clean.select_dtypes(include='number').columns

for col in numeric_cols:
    if df_clean[col].isnull().any():
        df_clean[col] = df_clean[col].fillna(
            df_clean[col].median()
        )

# 3. Check missing values after treatment
print("Missing values after treatment:")
print(df_clean.isnull().sum())

print("Total missing values:", df_clean.isnull().sum().sum())

In [ ]:
# Check whether missing values have been treated
missing_values_treated = df_clean.isnull().sum().sum() == 0

print("Have all missing values been treated?",
      missing_values_treated)

In [ ]:
print(df_clean.isnull().sum())

In [ ]:
numerical_cols = df.select_dtypes(include=['number']).columns

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower_limit) |
        (df[col] > upper_limit)
    ][col]

    print(f"\nColumn: {col}")
    print(f"Number of outliers: {len(outliers)}")

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

# Select numerical columns
numerical_cols = df.select_dtypes(include=['number']).columns

# Create boxplots
plt.figure(figsize=(12, 6))

for i, col in enumerate(numerical_cols, 1):
    plt.subplot(2, 4, i)
    sns.boxplot(y=df[col])
    plt.title(col)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

df_clean = df_clean.copy()

# Identify continuous numerical columns
binary_cols = ['telecommuting', 'has_company_logo',
               'has_questions', 'fraudulent', 'id']
num_cols = df_clean.select_dtypes(include='number').columns
cols = [c for c in num_cols if c not in binary_cols]

# Treat outliers using IQR clipping
for c in cols:
    Q1, Q3 = df_clean[c].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    df_clean[c] = df_clean[c].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

# Verify results
print("Missing values:", df_clean.isnull().sum().sum())
print("Shape:", df_clean.shape)
print("Binary columns preserved:",
      all(df_clean[c].dropna().isin([0, 1]).all()
          for c in binary_cols if c in df_clean.columns))

In [ ]:
df_clean = df.copy()

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder

df_model = df_clean.copy()

# Encode categorical columns
cat_cols = df_model.select_dtypes(include=['object', 'category']).columns
df_model[cat_cols] = df_model[cat_cols].astype(str)
df_model = pd.get_dummies(df_model, columns=cat_cols, dtype=int)

# Scale numerical columns
num_cols = df_model.select_dtypes(include='number').columns.drop(
    ['id', 'fraudulent'], errors='ignore'
)
df_model[num_cols] = StandardScaler().fit_transform(df_model[num_cols])

# Verify
print("Encoding and scaling completed:", df_model.select_dtypes(
    include=['object', 'category']
).empty)
print(df_model.shape)
print(df_model.head())

In [ ]:
from sklearn.model_selection import train_test_split

# Separate features and label
X = df_model.drop(columns=['fraudulent', 'id'], errors='ignore')
y = df_model['fraudulent']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Verify
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

In [ ]:


from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

models = {
    "Logistic Regression (sigmoid)": LogisticRegression(
        max_iter=1000, class_weight="balanced"
    ),
    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, class_weight="balanced",
        random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=50, random_state=42
    )
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-score": f1_score(y_test, pred, zero_division=0)
    })

print(pd.DataFrame(results).sort_values("F1-score", ascending=False))

In [ ]:
# Leak-safe word + character TF-IDF model with structured features.
# Select C and the fraud threshold on validation data; keep the holdout untouched.
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_recall_curve, precision_score, recall_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

text_columns = [
    column for column in ["title", "company_profile", "description", "requirements", "benefits"]
    if column in df.columns
]
category_columns = [
    column for column in [
        "location", "department", "salary_range", "employment_type",
        "required_experience", "required_education", "industry", "function",
    ]
    if column in df.columns
]
numeric_columns = [
    column for column in ["telecommuting", "has_company_logo", "has_questions"]
    if column in df.columns
]

accuracy_data = df.dropna(subset=["fraudulent"]).copy()
# Add lightweight signals about how much information each text field contains.
length_columns = [
    feature
    for column in text_columns
    for feature in (f"{column}_log_chars", f"{column}_log_words")
]
numeric_columns.extend(length_columns)
feature_columns = ["combined_text"] + category_columns + numeric_columns

def prepare_job_features(raw_frame):
    prepared = raw_frame.copy()
    text_data = {}
    for column in text_columns:
        if column not in prepared:
            prepared[column] = ""
        text_data[column] = prepared[column].fillna("").astype(str)
        prepared[f"{column}_log_chars"] = np.log1p(text_data[column].str.len())
        prepared[f"{column}_log_words"] = np.log1p(
            text_data[column].str.split().str.len()
        )
    prepared["combined_text"] = pd.DataFrame(text_data, index=prepared.index).agg(
        " ".join, axis=1
    )
    return prepared.reindex(columns=feature_columns)

X_accuracy = prepare_job_features(accuracy_data)
y_accuracy = accuracy_data["fraudulent"].astype(int)

# Keep a final holdout set untouched while selecting model settings.
X_development, X_test_accuracy, y_development, y_test_accuracy = train_test_split(
    X_accuracy, y_accuracy, test_size=0.2, random_state=42, stratify=y_accuracy
)
X_fit, X_validation, y_fit, y_validation = train_test_split(
    X_development, y_development, test_size=0.25, random_state=42,
    stratify=y_development,
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "text",
            FeatureUnion([
                (
                    "word",
                    TfidfVectorizer(
                        ngram_range=(1, 2), min_df=2, max_features=150000,
                        sublinear_tf=True, strip_accents="unicode",
                    ),
                ),
                (
                    "character",
                    TfidfVectorizer(
                        analyzer="char_wb", ngram_range=(3, 5), min_df=2,
                        max_features=100000, sublinear_tf=True,
                        strip_accents="unicode",
                    ),
                ),
            ]),
            "combined_text",
        ),
        (
            "categories",
            Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
                ("one_hot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            category_columns,
        ),
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scale", StandardScaler(with_mean=False)),
            ]),
            numeric_columns,
        ),
    ],
    remainder="drop",
)
# Cross-validate the complete preprocessing + model pipeline on the fit split.
# The validation and final holdout rows are never used to fit vectorizers or encoders.
candidate_pipeline = Pipeline([
    ("features", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000, solver="liblinear", random_state=42,
    )),
])
cross_validation = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model_search = GridSearchCV(
    estimator=candidate_pipeline,
    param_grid={
        "classifier__C": [0.25, 1.0, 4.0],
        "classifier__class_weight": [None, "balanced"],
    },
    scoring="f1",
    cv=cross_validation,
    n_jobs=1,
    refit=True,
    return_train_score=False,
)
model_search.fit(X_fit, y_fit)
best_model = model_search.best_estimator_
best_c = model_search.best_params_["classifier__C"]
best_class_weight = model_search.best_params_["classifier__class_weight"]

# Select the operating threshold on validation data, not on the final holdout.
validation_probabilities = best_model.predict_proba(X_validation)[:, 1]
precision_values, recall_values, thresholds = precision_recall_curve(
    y_validation, validation_probabilities
)
validation_f1_values = (
    2 * precision_values[:-1] * recall_values[:-1]
    / (precision_values[:-1] + recall_values[:-1] + 1e-12)
)
threshold_index = int(np.argmax(validation_f1_values))
best_threshold = float(thresholds[threshold_index])
best_validation_f1 = float(validation_f1_values[threshold_index])
validation_predictions = (validation_probabilities >= best_threshold).astype(int)

# Compare fit-split performance with cross-validation to flag a generalization gap.
fit_predictions = best_model.predict(X_fit)
fit_f1_at_default_threshold = f1_score(y_fit, fit_predictions, zero_division=0)
cv_f1 = float(model_search.best_score_)
print(f"5-fold CV F1 at threshold 0.5: {cv_f1:.3f}")
print(f"Fit-split F1 at threshold 0.5: {fit_f1_at_default_threshold:.3f}")
print(f"Fit/CV F1 gap: {fit_f1_at_default_threshold - cv_f1:.3f}")
print(f"Best parameters: {model_search.best_params_}")
print(f"Validation F1 at selected threshold: {best_validation_f1:.3f}")

# Report once on the untouched holdout set.
test_probabilities = best_model.predict_proba(X_test_accuracy)[:, 1]
test_predictions = (test_probabilities >= best_threshold).astype(int)
majority_baseline = max(y_test_accuracy.mean(), 1 - y_test_accuracy.mean())
print(f"Majority-class baseline accuracy: {majority_baseline:.3f}")
print(f"Validation-selected threshold: {best_threshold:.3f}")
print(f"Validation precision: {precision_score(y_validation, validation_predictions, zero_division=0):.3f}")
print(f"Validation recall:    {recall_score(y_validation, validation_predictions, zero_division=0):.3f}")
print(f"Holdout accuracy:  {accuracy_score(y_test_accuracy, test_predictions):.3f}")
print(f"Holdout precision: {precision_score(y_test_accuracy, test_predictions, zero_division=0):.3f}")
print(f"Holdout recall:    {recall_score(y_test_accuracy, test_predictions, zero_division=0):.3f}")
print(f"Holdout F1:        {f1_score(y_test_accuracy, test_predictions):.3f}")
print("\nConfusion matrix:\n", confusion_matrix(y_test_accuracy, test_predictions))
print("\nClassification report:\n", classification_report(
    y_test_accuracy, test_predictions,
    target_names=["legitimate", "fraudulent"], zero_division=0,
))

# Refit the selected configuration on all labeled rows for later inference.
fraud_pipeline = Pipeline([
    ("features", preprocessor),
    ("classifier", LogisticRegression(
        class_weight=best_class_weight, max_iter=2000, C=best_c,
        solver="liblinear", random_state=42,
    )),
])
fraud_pipeline.fit(X_accuracy, y_accuracy)

# Save the trained pipeline and the inputs needed to prepare future rows.
import joblib
from pathlib import Path

model_dir = Path.cwd() / "model"
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / "fake_job_posting_pipeline.joblib"
joblib.dump(
    {
        "pipeline": fraud_pipeline,
        "threshold": best_threshold,
        "feature_columns": feature_columns,
        "text_columns": text_columns,
        "category_columns": category_columns,
        "numeric_columns": numeric_columns,
    },
    model_path,
    compress=3,
)
print(f"Saved trained model pipeline to: {model_path.resolve()}")


In [ ]:

import matplotlib.pyplot as plt
import pandas as pd

# Convert results into a DataFrame
results_df = pd.DataFrame(results)

# Plot model performance
plt.figure(figsize=(10, 5))

for metric in ['Accuracy', 'Precision', 'Recall', 'F1-score']:
    plt.plot(results_df['Model'], results_df[metric],
             marker='o', label=metric)

plt.title('Performance Comparison of Classification Models')
plt.xlabel('Models')
plt.ylabel('Score')
plt.xticks(rotation=20, ha='right')
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.regularizers import l2

# Binary classification neural network
model_relu_log = Sequential([
    Input(shape=(X_train.shape[1],)),

    Dense(64, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.5),

    Dense(32, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.5),

    Dense(16, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.5),

    # Binary classification output
    Dense(1, activation='sigmoid')
])

In [ ]:
# Display architecture
model_relu_log.summary()

In [ ]:
import numpy as np
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

# Compile model
model_relu_log.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Define callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-5,
        verbose=1
    )
]

history = model_relu_log.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

In [ ]:
import numpy as np

# Use the validation-tuned TF-IDF model from the fraud-accuracy-pipeline cell.
y_pred_prob = np.asarray(test_probabilities).ravel()
y_pred = np.asarray(test_predictions).ravel().astype(int)

# Display prediction statistics
print("Minimum fraud probability:", np.min(y_pred_prob))
print("Maximum fraud probability:", np.max(y_pred_prob))
print("Mean fraud probability:", np.mean(y_pred_prob))

print("Predicted legitimate jobs (0):", np.sum(y_pred == 0))
print("Predicted fraudulent jobs (1):", np.sum(y_pred == 1))

In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# Evaluate the validation-tuned TF-IDF model on its untouched holdout set.
y_pred_prob = np.asarray(test_probabilities).ravel()
y_pred = np.asarray(test_predictions).ravel().astype(int)
y_true = np.asarray(y_test_accuracy).ravel().astype(int)

# Evaluate the model
print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall:", recall_score(y_true, y_pred, zero_division=0))
print("F1-score:", f1_score(y_true, y_pred, zero_division=0))
print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred))
print("\nClassification Report:\n",
      classification_report(y_true, y_pred, zero_division=0))

In [ ]:

import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training Loss vs Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# Load competition test data
test = pd.read_csv(
    "/kaggle/input/competitions/fake-job-posting/job_postings_test.csv"
)

# Make a copy
test_data = test.copy()

# Check dimensions
print("Test shape:", test_data.shape)

# Display columns
print(test_data.columns.tolist())

# Check missing values
print(test_data.isnull().sum())

In [ ]:
test_ids = test_data['id'].copy()

# Create test features
test_features = test_data.drop(columns=['id'])

print("Test features shape:", test_features.shape)
print("Test IDs shape:", test_ids.shape)

In [ ]:
print("Training columns:")
print(df.columns.tolist())

print("\nCompetition test columns:")
print(test_data.columns.tolist())

In [ ]:
test_ids = test_data['id'].copy()

In [ ]:
# Create the cleaned test dataset
test_clean = test_data.copy()

# Convert object columns to category
test_clean[test_clean.select_dtypes('object').columns] = (
    test_clean.select_dtypes('object').astype('category')
)

# Verify conversion
print(test_clean.dtypes)
print("Success:", not any(test_clean.dtypes == 'object'))

In [ ]:
test_ids = test_data['id'].copy()

# Create test features
test_features = test_data.drop(columns=['id'])

print("Test features shape:", test_features.shape)
print("Test IDs shape:", test_ids.shape)

In [ ]:
# Make a copy of the competition test dataset
test_clean = test_data.copy()

# 1. Fill missing categorical and text columns
text_cols = [
    'title',
    'location',
    'department',
    'salary_range',
    'company_profile',
    'description',
    'requirements',
    'benefits',
    'employment_type',
    'required_experience',
    'required_education',
    'industry',
    'function'
]

for col in text_cols:
    if col in test_clean.columns:
        test_clean[col] = test_clean[col].fillna('Unknown')

# 2. Fill missing numerical columns
numeric_cols = test_clean.select_dtypes(include='number').columns

for col in numeric_cols:
    if test_clean[col].isnull().any():
        test_clean[col] = test_clean[col].fillna(
            test_clean[col].median()
        )

# 3. Check missing values after treatment
print("Missing values after treatment:")
print(test_clean.isnull().sum())

print("\nTotal missing values:", test_clean.isnull().sum().sum())
print("\nTest data shape:", test_clean.shape)

In [ ]:
import pandas as pd

# Make a copy of competition test data
test_clean = test_clean.copy()

# Identify binary columns (fraudulent is not in competition test data)
binary_cols = [
    'telecommuting',
    'has_company_logo',
    'has_questions',
    'id'
]

# Identify continuous numerical columns
num_cols = test_clean.select_dtypes(include='number').columns
cols = [c for c in num_cols if c not in binary_cols]

# Treat outliers using IQR clipping
for c in cols:
    Q1, Q3 = test_clean[c].quantile([0.25, 0.75])
    IQR = Q3 - Q1

    test_clean[c] = test_clean[c].clip(
        Q1 - 1.5 * IQR,
        Q3 + 1.5 * IQR
    )

# Verify results
print("Missing values:", test_clean.isnull().sum().sum())
print("Shape:", test_clean.shape)

print(
    "Binary columns preserved:",
    all(
        test_clean[c].dropna().isin([0, 1]).all()
        for c in binary_cols if c in test_clean.columns
    )
)

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Create a copy of competition test data
test_model = test_clean.copy()

# Verify the dataset
print("Competition test data shape:", test_model.shape)
print(test_model.head())

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='most_frequent')

X_train_imputed = imputer.fit_transform(X_train)

In [ ]:
test_features = test_data.copy()

print("Competition test shape:", test_features.shape)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

decision_tree_model = DecisionTreeClassifier(
    random_state=42,
    class_weight='balanced'
)

decision_tree_model.fit(X_train, y_train)